# 角度归一化处理 (Gradient Descent / Least Squares)

本 Notebook 调用 `functions/angle_normalization.py` 模块完成夜光遥感数据的角度归一化处理。

**功能说明：**
- 读取时间序列数据
- 对每个像元进行传感器天顶角校正
- 输出校正后的时间序列及评价指标

In [ ]:
# 导入角度归一化模块
import sys
import os
import warnings
import matplotlib.pyplot as plt

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 添加项目路径（确保能导入 functions 模块）
cwd = os.getcwd()
project_root = cwd if os.path.basename(cwd) == 'ntl_prophet' else os.path.join(cwd, 'ntl_prophet')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 清除已缓存的模块以确保加载最新代码
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('functions'):
        del sys.modules[mod_name]

# 从 functions 模块导入函数
from functions.angle_normalization import (
    run_angle_normalization,
    display_comparison_results,
    plot_comparison_boxplot,
    plot_r2_distribution,
    readFile,
    visScatterAndFitCurve,
    visTimeSeries
)
from functions.timeseries_analysis import plot_mean_ntl_before_after

print('✅ 角度归一化模块加载成功')


## 配置参数

In [ ]:
# ============== 配置区域 ==============
# 输入输出路径
INPUT_DIR = r'.\processed'
OUTPUT_DIR = r'.\output'

# 要处理的数据集名称列表（不含 .txt 后缀）
NAME_LIST = ['ntl_timeseries']

# 可视化选项
PLOT_SCATTER = False   # 是否绘制散点图与拟合曲线
PLOT_SERIES = False    # 是否绘制时间序列折线图

# 并行化选项
USE_PARALLEL = True    # 是否使用多进程并行处理（大数据集推荐开启）
N_WORKERS = 16      # 并行进程数，None 表示自动使用 CPU核心数-1

# 归一化前异常值过滤（建议开启）
ENABLE_3SIGMA_FILTER = True
SIGMA_N = 3.0
NTL_UPPER_BOUND = 1000.0   # 本地区经验上限，可设为 None 关闭硬阈值
MIN_RECORDS_AFTER_FILTER = 10

print(f'输入目录: {INPUT_DIR}')
print(f'输出目录: {OUTPUT_DIR}')
print(f'待处理数据集: {NAME_LIST}')
print(f"并行处理: {'开启' if USE_PARALLEL else '关闭'}")
print(f"3-sigma过滤: {'开启' if ENABLE_3SIGMA_FILTER else '关闭'}, sigma={SIGMA_N}")
print(f"NTL上限阈值: {NTL_UPPER_BOUND if NTL_UPPER_BOUND is not None else '不限制'}")


## 方式一：一键运行（推荐）
使用 `run_angle_normalization()` 函数一键完成全流程。

In [ ]:
# 一键运行角度归一化（支持并行处理）
for name in NAME_LIST:
    input_file = os.path.join(INPUT_DIR, f'{name}.txt')

    if not os.path.exists(input_file):
        print(f'⚠️ 文件不存在: {input_file}')
        continue

    print(f'
{"="*50}')
    print(f'处理数据集: {name}')
    print(f'{"="*50}')

    results = run_angle_normalization(
        input_file=input_file,
        output_dir=OUTPUT_DIR,
        name=name,
        plot_scatter=PLOT_SCATTER,
        plot_series=PLOT_SERIES,
        verbose=False,
        show_progress=True,
        parallel=USE_PARALLEL,
        n_workers=N_WORKERS,
        prefilter_3sigma=ENABLE_3SIGMA_FILTER,
        prefilter_sigma=SIGMA_N,
        prefilter_ntl_upper_bound=NTL_UPPER_BOUND,
        min_records_after_filter=MIN_RECORDS_AFTER_FILTER
    )

    print(f'
📁 输出文件:')
    print(f'   校正结果: {results["fit_result_path"]}')
    print(f'   拟合R²: {results["output_r2_path"]}')
    print(f'   拟合参数: {results["output_params_path"]}')
    print(f'   有效点位数: {results["num_points"]}')

    pf = results.get('prefilter_summary', {})
    if pf.get('enabled', False):
        print('   异常值过滤统计:')
        print(f"      点位: {pf['points_before']} -> {pf['points_after']}")
        print(f"      记录: {pf['records_before']} -> {pf['records_after']}")
        print(f"      剔除记录: {pf['records_removed']}")

    display_comparison_results(results, name, show_details=10)
    plot_comparison_boxplot(results, name)
    plot_r2_distribution(results, name)

    mean_ts = plot_mean_ntl_before_after(
        original_file=input_file,
        normalized_file=results['fit_result_path'],
        title=f'{name} 归一化前后夜光均值时序'
    )
    print(f"   均值对比日期数: {mean_ts['stats']['n_dates']}")

print('
✅ 全部处理完成！')
